# Group Assignment 2 — Problem 2
## Mars Rover Sample Collection Planning using PDDL

**Group members:**

1. Siddharth — B24EC017
2. Parveen — B24EC013
3. Rakesh — B24EC023
4. Jnaneswar — B24ME027

This notebook runs the Fast Downward planner on `Domain.pddl` with `Problem.pddl` and `Problem_extended.pddl`, using two search configurations. It then checks each plan with VAL and with a step-by-step simulation, and prints the facts used in the plan analysis.

**Setup:** install Fast Downward (https://github.com/aibasel/downward) and VAL (https://github.com/KCL-Planning/VAL), then set the environment variables `FD_PATH` (path to `fast-downward.py`) and `VAL_PATH` (path to `Validate`). Planner outputs are saved to `planner_output/`.

In [1]:
import os, re, subprocess, time
import pandas as pd

FD  = os.environ.get("FD_PATH",  "fast-downward.py")
VAL = os.environ.get("VAL_PATH", "Validate")
DOMAIN = "Domain.pddl"
PROBLEMS = {"Base scenario": "Problem.pddl", "Extended scenario": "Problem_extended.pddl"}
CONFIGS = {
    "Satisficing (GBFS + FF)":     "eager_greedy([ff()])",   # greedy best-first search, FF heuristic
    "Optimal (A* + iPDB)":         "astar(ipdb())",          # A* with an admissible heuristic
}
OUT = "planner_output"
os.makedirs(OUT, exist_ok=True)

### Task 1 and Task 2: Domain and problem files

In [2]:
for f in [DOMAIN, *PROBLEMS.values()]:
    print("=" * 30, f, "=" * 30)
    print(open(f).read())

============================== Domain.pddl ==============================
;; Problem 2: Mars Rover Sample Collection - Domain
;; Uses only :strips and :typing (no numeric fluents, no negative preconditions)

(define (domain mars-rover)
  (:requirements :strips :typing)

  ;; ---------------- Types ----------------
  (:types
    rover location rock bay drill)

  ;; ---------------- Predicates ----------------
  (:predicates
    (rover-at ?rv - rover ?l - location)        ; rover is at location l
    (connected ?from - location ?to - location) ; rover can drive from -> to
    (rock-at ?r - rock ?l - location)           ; rock lies at location l
    (soft ?r - rock)                            ; rock can be collected directly
    (hard ?r - rock)                            ; rock needs the drill
    (drill-at ?d - drill ?l - location)         ; drill lies at location l
    (has-drill ?rv - rover)                     ; rover is carrying the drill
    (bay-of ?b - bay ?rv - rover)           

### Task 3: Plan generation (both configurations, both scenarios)

In [3]:
def run_planner(problem, search, tag):
    plan_file = os.path.join(OUT, f"{tag}.plan")
    t0 = time.time()
    res = subprocess.run(["python3", FD, "--plan-file", plan_file, DOMAIN, problem, "--search", search],
                         capture_output=True, text=True)
    wall = time.time() - t0
    log = res.stdout + res.stderr
    open(os.path.join(OUT, f"{tag}_planner_log.txt"), "w").write(log)
    grab = lambda pat: re.search(pat, log).group(1)
    plan = [l.strip() for l in open(plan_file) if l.startswith("(")]
    return {
        "plan": plan,
        "Plan length": int(grab(r"Plan length: (\d+) step")),
        "Plan cost": int(grab(r"Plan cost: (\d+)")),
        "States expanded": int(grab(r"Expanded (\d+) state")),
        "Search time (s)": float(grab(r"Search time: ([\d.]+)s")),
        "Planner total time (s)": float(grab(r"Total time: ([\d.]+)s")),
        "plan_file": plan_file,
    }

results = {}
for scen, prob in PROBLEMS.items():
    for cname, search in CONFIGS.items():
        tag = ("base" if prob == "Problem.pddl" else "extended") + ("_satisficing" if "Satisficing" in cname else "_optimal")
        results[(scen, cname)] = run_planner(prob, search, tag)
        print(f"done: {scen} / {cname}")

done: Base scenario / Satisficing (GBFS + FF)


done: Base scenario / Optimal (A* + iPDB)
done: Extended scenario / Satisficing (GBFS + FF)


done: Extended scenario / Optimal (A* + iPDB)


In [4]:
for (scen, cname), r in results.items():
    print("=" * 70)
    print(f"{scen}  |  {cname}  |  search = {CONFIGS[cname]}")
    print("=" * 70)
    for i, a in enumerate(r["plan"], 1):
        print(f"{i:3d}. {a}")
    print(f"Plan length: {r['Plan length']}   Total actions: {len(r['plan'])}   "
          f"States expanded: {r['States expanded']}   Search time: {r['Search time (s)']:.4f} s   "
          f"Total planner time: {r['Planner total time (s)']:.4f} s\n")

Base scenario  |  Satisficing (GBFS + FF)  |  search = eager_greedy([ff()])
  1. (move curiosity base sitea)
  2. (collect-soft curiosity rock1 bay1 sitea)
  3. (move curiosity sitea base)
  4. (deposit curiosity rock1 bay1 base)
  5. (move curiosity base siteb)
  6. (collect-soft curiosity rock2 bay1 siteb)
  7. (move curiosity siteb base)
  8. (deposit curiosity rock2 bay1 base)
  9. (move curiosity base sitea)
 10. (move curiosity sitea sitec)
 11. (move curiosity sitec sitee)
 12. (pick-drill curiosity drill1 sitee)
 13. (move curiosity sitee sitec)
 14. (collect-hard curiosity rock3 bay1 sitec)
 15. (move curiosity sitec sited)
 16. (collect-soft curiosity rock4 bay2 sited)
 17. (move curiosity sited siteb)
 18. (move curiosity siteb base)
 19. (deposit curiosity rock3 bay1 base)
 20. (deposit curiosity rock4 bay2 base)
 21. (move curiosity base sitea)
 22. (move curiosity sitea sitec)
 23. (move curiosity sitec sitee)
 24. (move curiosity sitee sitef)
 25. (collect-hard curiosity

In [5]:
summary = pd.DataFrame([
    {"Scenario": s, "Configuration": c, "Plan length": r["Plan length"], "Total actions": len(r["plan"]),
     "States expanded": r["States expanded"], "Search time (s)": round(r["Search time (s)"], 4),
     "Total time (s)": round(r["Planner total time (s)"], 4)}
    for (s, c), r in results.items()])
summary.to_csv(os.path.join(OUT, "summary.csv"), index=False)
summary

,Scenario,Configuration,Plan length,Total actions,States expanded,Search time (s),Total time (s)
0,Base scenario,Satisficing (GBFS + FF),30,30,51,0.0010,0.0042
1,Base scenario,Optimal (A* + iPDB),25,25,1930,0.0059,1.1225
2,Extended scenario,Satisficing (GBFS + FF),51,51,79,0.0014,0.0036
3,Extended scenario,Optimal (A* + iPDB),40,40,868407,1.9373,4.3092


### Task 4: Goal verification
**(a) VAL plan validator**

In [6]:
for (scen, cname), r in results.items():
    prob = PROBLEMS[scen]
    out = subprocess.run([VAL, DOMAIN, prob, r["plan_file"]], capture_output=True, text=True).stdout
    verdict = "Plan valid" if "Plan valid" in out else "NOT valid"
    print(f"{scen:18s} | {cname:26s} | VAL: {verdict}  ({'Successful plans' in out and 'goal satisfied' or ''})")

Base scenario      | Satisficing (GBFS + FF)    | VAL: Plan valid  (goal satisfied)
Base scenario      | Optimal (A* + iPDB)        | VAL: Plan valid  (goal satisfied)
Extended scenario  | Satisficing (GBFS + FF)    | VAL: Plan valid  (goal satisfied)
Extended scenario  | Optimal (A* + iPDB)        | VAL: Plan valid  (goal satisfied)


**(b) Step-by-step simulation.** Each action's preconditions are checked against the current state and its effects are applied. The final state is printed and checked against the goal.

In [7]:
def initial_state(problem_file):
    txt = open(problem_file).read().lower()
    init = txt[txt.index("(:init"):txt.index("(:goal")]
    init = re.sub(r";.*", "", init)
    facts = set(re.findall(r"\(([a-z][a-z0-9-]*(?: [a-z0-9-]+)+)\)", init))
    goal = txt[txt.index("(:goal"):]
    goal = re.sub(r";.*", "", goal)
    goals = set(re.findall(r"\(([a-z][a-z0-9-]*(?: [a-z0-9-]+)+)\)", goal))
    return facts, goals

def apply(state, action):
    name, *args = action.strip("()").split()
    if name == "move":
        rv, a, b = args
        pre = {f"rover-at {rv} {a}", f"connected {a} {b}"}
        add, dele = {f"rover-at {rv} {b}"}, {f"rover-at {rv} {a}"}
    elif name == "pick-drill":
        rv, d, l = args
        pre = {f"rover-at {rv} {l}", f"drill-at {d} {l}"}
        add, dele = {f"has-drill {rv}"}, {f"drill-at {d} {l}"}
    elif name in ("collect-soft", "collect-hard"):
        rv, r, b, l = args
        pre = {f"rover-at {rv} {l}", f"rock-at {r} {l}", f"bay-of {b} {rv}", f"empty {b}"}
        pre.add(f"soft {r}" if name == "collect-soft" else f"hard {r}")
        if name == "collect-hard":
            pre.add(f"has-drill {rv}")
        add, dele = {f"in-bay {r} {b}"}, {f"rock-at {r} {l}", f"empty {b}"}
    elif name == "deposit":
        rv, r, b, l = args
        pre = {f"rover-at {rv} {l}", f"in-bay {r} {b}", f"bay-of {b} {rv}"}
        add, dele = {f"rock-at {r} {l}", f"empty {b}"}, {f"in-bay {r} {b}"}
    missing = pre - state
    if missing:
        raise ValueError(f"{action}: precondition(s) not satisfied: {missing}")
    return (state - dele) | add

def show(state):
    rover = [f.split()[2] for f in state if f.startswith("rover-at")][0]
    bays = {b: ([f.split()[1] for f in state if f.startswith("in-bay") and f.endswith(b)] or ["empty"])[0]
            for b in ("bay1", "bay2")}
    drill = "carried" if "has-drill curiosity" in state else [f.split()[2] for f in state if f.startswith("drill-at")][0]
    return rover, bays, drill

def simulate(scen, cname, verbose=True):
    state, goals = initial_state(PROBLEMS[scen])
    if verbose: print(f"--- {scen} | {cname} ---")
    for i, a in enumerate(results[(scen, cname)]["plan"], 1):
        state = apply(state, a)
        if verbose:
            rover, bays, drill = show(state)
            print(f"{i:3d}. {a:45s} rover@{rover:6s} bay1={bays['bay1']:6s} bay2={bays['bay2']:6s} drill={drill}")
    return state, goals

for key in results:
    final, goals = simulate(*key, verbose=(key[1].startswith("Optimal")))
    rover, bays, drill = show(final)
    rocks = sorted(f for f in final if f.startswith("rock-at"))
    print(f"\nFINAL STATE [{key[0]} | {key[1]}]")
    print("  Rocks :", ", ".join(f"{f.split()[1]}@{f.split()[2]}" for f in rocks))
    print(f"  Bays  : bay1={bays['bay1']}, bay2={bays['bay2']}")
    print(f"  Rover : {rover}    Drill: {drill}")
    print("  All goal facts satisfied:", goals <= final, "\n")


FINAL STATE [Base scenario | Satisficing (GBFS + FF)]
  Rocks : rock1@base, rock2@base, rock3@base, rock4@base, rock5@base
  Bays  : bay1=empty, bay2=empty
  Rover : base    Drill: carried
  All goal facts satisfied: True 

--- Base scenario | Optimal (A* + iPDB) ---
  1. (move curiosity base siteb)                   rover@siteb  bay1=empty  bay2=empty  drill=sitee
  2. (collect-soft curiosity rock2 bay1 siteb)     rover@siteb  bay1=rock2  bay2=empty  drill=sitee
  3. (move curiosity siteb sited)                  rover@sited  bay1=rock2  bay2=empty  drill=sitee
  4. (collect-soft curiosity rock4 bay2 sited)     rover@sited  bay1=rock2  bay2=rock4  drill=sitee
  5. (move curiosity sited siteb)                  rover@siteb  bay1=rock2  bay2=rock4  drill=sitee
  6. (move curiosity siteb base)                   rover@base   bay1=rock2  bay2=rock4  drill=sitee
  7. (deposit curiosity rock2 bay1 base)           rover@base   bay1=empty  bay2=rock4  drill=sitee
  8. (deposit curiosity rock4 b

### Task 6: Facts used in the plan analysis

In [8]:
def trips(plan):
    """Split the plan into round trips that start and end at Base."""
    out, cur, loc = [], ["base"], "base"
    loaded = []
    for a in plan:
        name, *args = a.strip("()").split()
        if name == "move":
            loc = args[2]; cur.append(loc)
            if loc == "base":
                out.append((cur, loaded)); cur, loaded = ["base"], []
        elif name.startswith("collect"):
            loaded.append(args[1])
    return out

for key, r in results.items():
    plan = r["plan"]
    drill_step = next(i for i, a in enumerate(plan, 1) if a.startswith("(pick-drill"))
    print(f"{key[0]} | {key[1]}")
    print(f"  Drill picked up at step {drill_step}: {plan[drill_step-1]}")
    for n, (route, rocks) in enumerate(trips(plan), 1):
        print(f"  Trip {n}: {' -> '.join(route):60s} rocks collected: {', '.join(rocks) or '-'}")
    print(f"  One-way route SiteC->SiteD used: {'(move curiosity sitec sited)' in plan}\n")

Base scenario | Satisficing (GBFS + FF)
  Drill picked up at step 12: (pick-drill curiosity drill1 sitee)
  Trip 1: base -> sitea -> base                                        rocks collected: rock1
  Trip 2: base -> siteb -> base                                        rocks collected: rock2
  Trip 3: base -> sitea -> sitec -> sitee -> sitec -> sited -> siteb -> base rocks collected: rock3, rock4
  Trip 4: base -> sitea -> sitec -> sitee -> sitef -> sitee -> sitec -> sitea -> base rocks collected: rock5
  One-way route SiteC->SiteD used: True

Base scenario | Optimal (A* + iPDB)
  Drill picked up at step 12: (pick-drill curiosity drill1 sitee)
  Trip 1: base -> siteb -> sited -> siteb -> base                      rocks collected: rock2, rock4
  Trip 2: base -> sitea -> sitec -> sitee -> sitef -> sitee -> sitec -> sitea -> base rocks collected: rock5, rock3
  Trip 3: base -> sitea -> base                                        rocks collected: rock1
  One-way route SiteC->SiteD used: F